In [3]:
import pandas as pd
import json

from curation_tools.curation_tools import (
    CuratedDataset,
    ObsSchema,
    VarSchema,
    Experiment,
    download_file,
    upload_parquet_to_bq
)

# import logging
# logging.basicConfig(
#     level=logging.DEBUG,
#     format="%(asctime)s %(levelname)s %(name)s: %(message)s",
#     handlers=[
#         logging.FileHandler("curation.log"),
#         logging.StreamHandler(),  # keep console output too
#     ],
#     force=True,
# )

/usr/local/lib/python3.12/dist-packages/pandera/_pandas_deprecated.py:157: FutureWarning: Importing pandas-specific classes and functions from the
top-level pandera module will be **removed in a future version of pandera**.
If you're using pandera to validate pandas objects, we highly recommend updating
your import:

```
# old import
import pandera as pa

# new import
import pandera.pandas as pa
```

If you're using pandera to validate objects from other compatible libraries
like pyspark or polars, see the supported libraries section of the documentation
for more information on how to import pandera:

https://pandera.readthedocs.io/en/stable/supported_libraries.html

To disable this warning, set the environment variable:

```
export DISABLE_PANDERA_IMPORT_WARNING=True
```

  warnings.warn(_future_warning, FutureWarning)


# Download data


In [4]:
noncurated_path = "../non_curated/h5ad/frangieh_2021_raw.h5ad"
download_file(
    url="https://exampledata.scverse.org/pertpy/frangieh_2021_raw.h5ad",
    dest_path=noncurated_path
)

File ../non_curated/h5ad/frangieh_2021_raw.h5ad already exists. Skipping download.


# Initialise the dataset object

In [24]:
cur_data = CuratedDataset(
    obs_schema=ObsSchema,
    var_schema=VarSchema,
    exp_metadata_schema=Experiment,
    noncurated_path=noncurated_path
)

cur_data.load_data()

Loading data from ../non_curated/h5ad/frangieh_2021_raw.h5ad


Filter the data on MOI == 1. This keeps only cells targeted with 1 guideRNA.

In [29]:
cur_data.adata = cur_data.adata[cur_data.adata.obs['MOI'] == "1"]

# OBS slot curation

### Rename sgRNA to perturbation_name

In [30]:
cur_data.rename_columns(slot = 'obs', name_dict = {'sgRNA': 'perturbation_name'})
cur_data.adata.obs

Renamed columns in adata.obs: {'sgRNA': 'perturbation_name'}


,library_preparation_protocol,condition,MOI,perturbation_name,UMI_count
cell_ID,,,,,
CELL_1,10X 3' v3 sequencing,Control,1,HLA-B_2,10832.0
CELL_3,10X 3' v3 sequencing,Control,1,HLA-B_2,28821.0
CELL_6,10X 3' v3 sequencing,Control,1,IFNGR1_1,8810.0
CELL_8,10X 3' v3 sequencing,Control,1,CDKN1A_3,8491.0
CELL_9,10X 3' v3 sequencing,Control,1,EMP1_3,7478.0
...,...,...,...,...,...
CELL_218323,10X 3' v3 sequencing,Co-culture,1,SLC22A18_2,15823.0
CELL_218325,10X 3' v3 sequencing,Co-culture,1,SLC19A1_2,30696.0
CELL_218326,10X 3' v3 sequencing,Co-culture,1,AHNAK_2,16010.0


### Show unique perturbations

In [31]:
cur_data.show_unique(slot = 'obs', column = 'perturbation_name')

Unique values in adata.obs.perturbation_name: 818
--------------------------------------------------
{'A2M_1',
 'A2M_2',
 'A2M_3',
 'ACSL3_1',
 'ACSL3_2',
 'ACSL3_3',
 'ACTA2_1',
 'ACTA2_2',
 'ACTA2_3',
 'AEBP1_1',
 'AEBP1_2',
 'AEBP1_3',
 'AGA_1',
 'AGA_2',
 'AGA_3',
 'AHCY_1',
 'AHCY_2',
 'AHCY_3',
 'AHNAK_1',
 'AHNAK_2',
 'AHNAK_3',
 'APOC2_1',
 'APOC2_2',
 'APOC2_3',
 'APOD_1',
 'APOD_2',
 'APOD_3',
 'APOE_1',
 'APOE_2',
 'APOE_3',
 'ARMC6_1',
 'ARMC6_2',
 'ARMC6_3',
 'ATP1A1_1',
 'ATP1A1_2',
 'ATP1A1_3',
 'ATP1B1_1',
 'ATP1B1_2',
 'ATP1B1_3',
 'ATP5MD_1',
 'ATP5MD_2',
 'ATP5MD_3',
 'B2M_1',
 'B2M_2',
 'B2M_3',
 'BOLA2B_1',
 'BOLA2B_2',
 'BOLA2B_3',
 'BOLA2_1',
 'BOLA2_2',
 'BOLA2_3',
 'BZW2_1',
 'BZW2_2',
 'BZW2_3',
 'C19orf48_1',
 'C19orf48_2',
 'C19orf48_3',
 'C1QBP_1',
 'C1QBP_2',
 'C1QBP_3',
 'C6orf226_1',
 'C6orf226_2',
 'C6orf226_3',
 'CCND1_1',
 'CCND1_2',
 'CCND1_3',
 'CCND2_1',
 'CCND2_2',
 'CCND2_3',
 'CCR10_1',
 'CCR10_2',
 'CCR10_3',
 'CCT3_1',
 'CCT3_2',
 'CCT3_3',
 '

### Add guide RNA information

In [32]:
download_file(
    url="https://static-content.springer.com/esm/art%3A10.1038%2Fs41588-021-00779-1/MediaObjects/41588_2021_779_MOESM3_ESM.xlsx",
    dest_path="../supplementary/frangieh_2021_supp.xlsx"
)

guide_info_df = pd.read_excel("../supplementary/frangieh_2021_supp.xlsx", sheet_name="Supplementary Table 1", skiprows=2)
guide_info_df = guide_info_df.rename(columns = {'Guide Name':'perturbation_name','sgRNA Sequence':'guide_sequence'})
guide_info_df

File ../supplementary/frangieh_2021_supp.xlsx already exists. Skipping download.


,perturbation_name,guide_sequence
0,A2M_1,TGAAATGAAACTTCACACTG
1,A2M_2,GGAAAAATGGCCCTTCACTG
2,A2M_3,ACTGCATCTGTGCAAACGGG
3,ACSL3_1,GTGGTGAAGAGTAACCAATG
4,ACSL3_2,TTTACTTACCTGTCGCACAG
...,...,...
1498,NO_SITE_821,TGTACGCTCCCGTCGAACGA
1499,NO_SITE_898,AAGCGGTTGTACGACGCACG
1500,NO_SITE_132,TACTCGTCGTACGTCGTTCG
1501,NO_SITE_439,CGGGGTCGGCATACGATCGG


In [33]:
cur_data.adata.obs = cur_data.adata.obs.merge(guide_info_df, on='perturbation_name', how='left')
cur_data.adata.obs

/usr/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


,library_preparation_protocol,condition,MOI,perturbation_name,UMI_count,guide_sequence
0,10X 3' v3 sequencing,Control,1,HLA-B_2,10832.0,CTCCGCAGGGTAGAAACCCA
1,10X 3' v3 sequencing,Control,1,HLA-B_2,28821.0,CTCCGCAGGGTAGAAACCCA
2,10X 3' v3 sequencing,Control,1,IFNGR1_1,8810.0,GCCGCGAACGACGGTACCTG
3,10X 3' v3 sequencing,Control,1,CDKN1A_3,8491.0,AGTCGAAGTTCCATCGCTCA
4,10X 3' v3 sequencing,Control,1,EMP1_3,7478.0,CTCACAGCACACCAGTGTGG
...,...,...,...,...,...,...
126961,10X 3' v3 sequencing,Co-culture,1,SLC22A18_2,15823.0,GGCCTTCAGGTCGAACACAC
126962,10X 3' v3 sequencing,Co-culture,1,SLC19A1_2,30696.0,GGCCCGACAAGAACTTCACG
126963,10X 3' v3 sequencing,Co-culture,1,AHNAK_2,16010.0,CCCAGCTGCTGAACACCATG
126964,10X 3' v3 sequencing,Co-culture,1,CDKN2B_3,17941.0,CCGGTCGGGTGAGAGTGGCA


### Standardise perturbation targets

Two types of controls are used:
1. Non-targeting controls: `NO_SITE_*`
2. Controls targeting intergenic regions: `ONE_NON-GENE_SITE_*`

In [34]:
cur_data.adata.obs['target'] = (cur_data.adata.obs['perturbation_name']
    .str.replace('NO_SITE', 'control_nontargeting') # rename nontargeting control guides
    .str.replace('ONE_NON-GENE_SITE', 'control_intergenic') # rename intergenic control guides
    .str.replace(r"_\d+", "", regex=True) # remove guide number suffix
    .fillna('control_casonly')
)

In [35]:
cur_data.standardize_genes(
    slot='obs',
    input_column='target',
    input_column_type='gene_symbol',
    multiple_entries=False,
    # multiple_entries_sep='|'
)

Mapping gene symbols: 100%|████████████████████████████████████| 250/250 [00:00<00:00, 20653.86it/s]
/usr/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


--------------------------------------------------
Successfully mapped 250 out of 250 gene symbols.
--------------------------------------------------
Couldn't map gene symbols: []
--------------------------------------------------


### Add `perturbed_target_number` column

In [36]:
cur_data.count_entries(
    slot='obs',
    input_column='perturbed_target_symbol',
    count_column_name='perturbed_target_number',
    sep='|'
)

Counted entries in column perturbed_target_symbol of adata.obs and stored in perturbed_target_number


### Encode chromosomes as integers

In [37]:
cur_data.chromosome_encoding()

Chromosome encoding applied to perturbed_target_chromosome in adata.obs and stored as 'perturbed_target_chromosome_encoding'.


### Add treatment information

In [38]:
cur_data.adata.obs['treatment_label'] = cur_data.adata.obs['condition'].replace({
    'Control': 'untreated control',
    'IFNγ': 'Interferon gamma',
    'Co-culture': 'Interferon gamma|Tumor Infiltrating Lymphocyte'
})

cur_data.adata.obs['treatment_id'] = cur_data.adata.obs['condition'].replace({
    'Control': 'NCIT:C184729',
    'IFNγ': 'CHEMBL:3286073',
    'Co-culture': 'CHEMBL:3286073|NCIT:C12546'
})

/tmp/ipython-input-3727342723.py:1: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  cur_data.adata.obs['treatment_label'] = cur_data.adata.obs['condition'].replace({
/tmp/ipython-input-3727342723.py:7: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  cur_data.adata.obs['treatment_id'] = cur_data.adata.obs['condition'].replace({


### Add timepoint information

In [39]:
cur_data.adata.obs['timepoint'] = cur_data.adata.obs['condition'].replace({
    'Control': 'P14DT16H0M0S',
    'IFNγ': 'P14DT16H0M0S',
    'Co-culture': 'P17DT16H0M0S'
})

/tmp/ipython-input-2328709114.py:1: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  cur_data.adata.obs['timepoint'] = cur_data.adata.obs['condition'].replace({


### Add metadata

In [40]:
cur_data.create_columns(
    overwrite=True,
    slot="obs",
    col_dict={
        #----- dataset -----#
        "dataset_id": cur_data.dataset_id,
        #----- sample -----#
        "sample_id": range(1, cur_data.adata.obs.shape[0] + 1),
        #----- perturbation type -----#
        "perturbation_type_label": "CRISPRn",
        "perturbation_type_id": None,
        #----- data modality -----#
        "data_modality": "Perturb-seq", # different from "method_name_label"; more general term - choice of CRISPR, MAVE and Perturb-seq
        #----- significance -----#
        "significant": None,
        "significance_criteria": None,
        #----- score interpretation -----#
        "score_interpretation": None,
        #----- treatment -----#
        # "treatment_label": None,
        # "treatment_id": None,
        #----- replicate -----#
        "technical_replicate": None,
        "biological_replicate": None,
        #----- model system -----#
        "model_system_label": "primary_cell",
        "model_system_id": None,
        #----- tissue -----#
        "tissue": "skin of body",
        #----- cell line -----#
        "cell_line_label": None,
        "cell_line_id": None,
        #----- cell type -----#
        "cell_type_label": "melanoma cell",
        "cell_type_id": "BTO:0000848",
        #----- disease -----#
        "disease_label": "melanoma",
        "disease_id": "MONDO:0005105",
        #----- timepoint -----#
        # "timepoint": "P7DT0H0M0S",
        #----- species -----#
        "species": "Homo sapiens",
        #----- sex -----#
        "sex_label": None,
        "sex_id": None,
        #----- developmental stage -----#
        "developmental_stage_label": None,
        "developmental_stage_id": None,
        #----- study metadata -----#
        "study_title": "Multimodal pooled Perturb-CITE-seq screens in patient models define mechanisms of cancer immune evasion",
        "study_uri": "https://doi.org/10.1038/s41588-021-00779-1",
        "study_year": 2021,
        #----- authors -----#
        "first_author": "Chris J. Frangieh",
        "last_author": "Benjamin Izar",
        #----- experiment metadata -----#
        "experiment_title": "Perturb-seq CRISPRko screen of primary melanoma cells in untreated, IFNg-stimulated and IFNg-stimulated + co-cultured with autologous tumor-infiltrating lymphocytes to explore cancer cell-intrinsic immune checkpoint inhibitor resistance mechanisms.",
        "experiment_summary": """
            Patient-derived melanoma cells stably expressing Cas9 were engineered by lentiviral transduction of a pooled CROPseq-mKate2 sgRNA library targeting 248 immunotherapy resistance genes. Transduced cells were cultured under antibiotic selection for 14 days, pre-treated with IFN-γ for 16 hours, and subsequently subjected to control, IFN-γ treatment, or co-culture with autologous tumor-infiltrating lymphocytes (TILs) conditions for 48 hours. At day 17, surviving cancer cells were harvested, stained with oligonucleotide-conjugated antibodies, sorted to remove TILs, processed using the Chromium Single Cell 3’ Library and Gel Bead kit v3, and sequenced on an Illumina HiSeq.
        """,
        #----- number of perturbed targets/samples -----#
        "number_of_perturbed_targets": len(set(cur_data.adata.obs['perturbed_target_coord'])),
        "number_of_perturbed_samples": cur_data.adata.obs.shape[0],
        #----- library generation type -----#
        "library_generation_type_id": "EFO:0022868",
        "library_generation_type_label": "endogenous",
        #----- library generation method -----#
        "library_generation_method_id": "EFO:0022876",
        "library_generation_method_label": "SpCas9",
        #----- enzyme and library delivery method -----#
        "enzyme_delivery_method_id": None,
        "enzyme_delivery_method_label": "lentivirus transduction",

        "library_delivery_method_id": None,
        "library_delivery_method_label": "lentivirus transduction",
        #----- enzyme and library integration state -----#
        "enzyme_integration_state_id": None,
        "enzyme_integration_state_label": "random locus integration",

        "library_integration_state_id": None,
        "library_integration_state_label": "random locus integration",
        #----- enzyme and library expression control -----#
        "enzyme_expression_control_id": None,
        "enzyme_expression_control_label": "constitutive transgene expression",

        "library_expression_control_id": None,
        "library_expression_control_label": "constitutive transgene expression",
        #----- library name and URI and manufacturer -----#
        "library_name": "custom",
        "library_uri": None,
        "library_manufacturer": "Izar lab",
        #----- library format -----#
        "library_format_id": None,
        "library_format_label": "pooled",
        #----- library scope -----#
        "library_scope_id": None,
        "library_scope_label": "focused",
        #----- library perturbation type -----#
        "library_perturbation_type_id": None,
        "library_perturbation_type_label": "knockout",
        #----- library additional metadata -----#
        "library_lentiviral_generation": "2",
        "library_grnas_per_target": "3",
        "library_total_grnas": str(cur_data.adata.obs['guide_sequence'].str.split('|').explode().nunique()), # for CRISPR/Perturb-seq
        "library_total_variants": None, # for MAVE
        #----- readout dimensionality -----#
        "readout_dimensionality_id": None,
        "readout_dimensionality_label": "high-dimensional assay",
        #---- readout type -----#
        "readout_type_id": None,
        "readout_type_label": "transcriptomic",
        #----- readout technology -----#
        "readout_technology_id": None,
        "readout_technology_label": "single-cell rna-seq",
        #----- method -----#
        "method_name_id": None,
        "method_name_label": "Perturb-CITE-seq", # different from "data_modality"; more specific term - specific name of the technique
        "method_uri": None,
        #----- sequencing library kit -----#
        "sequencing_library_kit_id": None,
        "sequencing_library_kit_label": "10x Genomics Single Cell 3-prime v3",
        #----- sequencing platform -----#
        "sequencing_platform_id": None,
        "sequencing_platform_label": "Illumina HiSeq 2500",
        #----- sequencing strategy -----#
        "sequencing_strategy_id": None,
        "sequencing_strategy_label": "barcode sequencing",
        #----- software used for counts-----#
        "software_counts_id": None,
        "software_counts_label": "CellRanger",
        #----- software used for analysis -----#
        "software_analysis_id": None,
        "software_analysis_label": "MAST",
        #----- reference genome -----#
        "reference_genome_id": None,
        "reference_genome_label": "GRCh38",
        #----- license -----#
        "license_label": "MIT License",
        "license_id": "SWO:9000074",
        #----- external datasets -----#
        "associated_datasets": json.dumps([
            {
                "dataset_accession": "frangieh_2021_raw.h5ad",
                "dataset_uri": "https://scverse-exampledata.s3.eu-west-1.amazonaws.com/pertpy/frangieh_2021_raw.h5ad",
                "dataset_description": "Raw counts - .h5ad file from pertpy",
                "dataset_file_name": "frangieh_2021_raw.h5ad",
            }
        ])
    }
)

Column dataset_id added to adata.obs
Column sample_id added to adata.obs
Column perturbation_type_label added to adata.obs
Column perturbation_type_id added to adata.obs
Column data_modality added to adata.obs
Column significant added to adata.obs
Column significance_criteria added to adata.obs
Column score_interpretation added to adata.obs
Column technical_replicate added to adata.obs
Column biological_replicate added to adata.obs
Column model_system_label added to adata.obs
Column model_system_id added to adata.obs
Column tissue added to adata.obs
Column cell_line_label added to adata.obs
Column cell_line_id added to adata.obs
Column cell_type_label added to adata.obs
Column cell_type_id added to adata.obs
Column disease_label added to adata.obs
Column disease_id added to adata.obs
Column species added to adata.obs
Column sex_label added to adata.obs
Column sex_id added to adata.obs
Column developmental_stage_label added to adata.obs
Column developmental_stage_id added to adata.obs
C

### Curate tissue information


In [41]:
cur_data.standardize_ontology(
    input_column='tissue',
    column_type='term_name',
    ontology_type='tissue',
    overwrite=True
)

Mapped 1 tissue ontology terms from `tissue` column to ontology terms
DataFrame shape: (1, 4)
--------------------------------------------------
   input_column input_column_lower    name_lower     ontology_id
0  skin of body       skin of body  skin of body  UBERON:0002097
--------------------------------------------------


/usr/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


### Match schema column order

In [42]:
cur_data.match_schema_columns(slot='obs')

Matched columns of adata.obs to the obs_schema.


### Validate obs metadata

In [43]:
cur_data.validate_data(slot='obs', verbose=True)

,dataset_id,sample_id,data_modality,significant,significance_criteria,perturbation_name,perturbed_target_coord,perturbed_target_chromosome,perturbed_target_chromosome_encoding,perturbed_target_number,...,software_counts_id,software_counts_label,software_analysis_id,software_analysis_label,score_interpretation,reference_genome_id,reference_genome_label,associated_datasets,license_label,license_id
0,frangieh_2021_raw,1,Perturb-seq,<NA>,<NA>,HLA-B_2,chr6:31353867-31367067;-1,6,6,1,...,<NA>,CellRanger,<NA>,MAST,<NA>,<NA>,GRCh38,"[{""dataset_accession"": ""frangieh_2021_raw.h5ad...",MIT License,SWO:9000074
1,frangieh_2021_raw,2,Perturb-seq,<NA>,<NA>,HLA-B_2,chr6:31353867-31367067;-1,6,6,1,...,<NA>,CellRanger,<NA>,MAST,<NA>,<NA>,GRCh38,"[{""dataset_accession"": ""frangieh_2021_raw.h5ad...",MIT License,SWO:9000074
2,frangieh_2021_raw,3,Perturb-seq,<NA>,<NA>,IFNGR1_1,chr6:137197483-137219449;-1,6,6,1,...,<NA>,CellRanger,<NA>,MAST,<NA>,<NA>,GRCh38,"[{""dataset_accession"": ""frangieh_2021_raw.h5ad...",MIT License,SWO:9000074
3,frangieh_2021_raw,4,Perturb-seq,<NA>,<NA>,CDKN1A_3,chr6:36676441-36687397;1,6,6,1,...,<NA>,CellRanger,<NA>,MAST,<NA>,<NA>,GRCh38,"[{""dataset_accession"": ""frangieh_2021_raw.h5ad...",MIT License,SWO:9000074
4,frangieh_2021_raw,5,Perturb-seq,<NA>,<NA>,EMP1_3,chr12:13196723-13219941;1,12,12,1,...,<NA>,CellRanger,<NA>,MAST,<NA>,<NA>,GRCh38,"[{""dataset_accession"": ""frangieh_2021_raw.h5ad...",MIT License,SWO:9000074
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
126961,frangieh_2021_raw,126962,Perturb-seq,<NA>,<NA>,SLC22A18_2,chr11:2899718-2925247;1,11,11,1,...,<NA>,CellRanger,<NA>,MAST,<NA>,<NA>,GRCh38,"[{""dataset_accession"": ""frangieh_2021_raw.h5ad...",MIT License,SWO:9000074
126962,frangieh_2021_raw,126963,Perturb-seq,<NA>,<NA>,SLC19A1_2,chr21:45493572-45573564;-1,21,21,1,...,<NA>,CellRanger,<NA>,MAST,<NA>,<NA>,GRCh38,"[{""dataset_accession"": ""frangieh_2021_raw.h5ad...",MIT License,SWO:9000074
126963,frangieh_2021_raw,126964,Perturb-seq,<NA>,<NA>,AHNAK_2,chr11:62433542-62556235;-1,11,11,1,...,<NA>,CellRanger,<NA>,MAST,<NA>,<NA>,GRCh38,"[{""dataset_accession"": ""frangieh_2021_raw.h5ad...",MIT License,SWO:9000074
126964,frangieh_2021_raw,126965,Perturb-seq,<NA>,<NA>,CDKN2B_3,chr9:22002903-22009305;-1,9,9,1,...,<NA>,CellRanger,<NA>,MAST,<NA>,<NA>,GRCh38,"[{""dataset_accession"": ""frangieh_2021_raw.h5ad...",MIT License,SWO:9000074


# VAR slot curation

### Standardise genes

In [48]:
cur_data.adata.var['gene_name'] = cur_data.adata.var.index
cur_data.adata.var

,gene_name
gene,
A1BG,A1BG
A1BG-AS1,A1BG-AS1
A1CF,A1CF
A2M,A2M
A2M-AS1,A2M-AS1
...,...
ZXDC,ZXDC
ZYG11A,ZYG11A
ZYG11B,ZYG11B


In [64]:
cur_data.standardize_genes(
    slot="var",
    input_column="gene_name",
    input_column_type="gene_symbol",
    remove_version=True,
    multiple_entries=False
)

Removed version numbers from gene_name


Mapping gene symbols: 100%|████████████████████████████████| 22474/22474 [00:02<00:00, 10987.32it/s]


--------------------------------------------------
Successfully mapped 21658 out of 22474 gene symbols.
--------------------------------------------------
Couldn't map gene symbols: ['AC002074', 'AC002384', 'AC002386', 'AC002454', 'AC002463', 'AC003084', 'AC003092', 'AC003975', 'AC003984', 'AC003988', 'AC004012', 'AC004158', 'AC004556', 'AC004870', 'AC004930', 'AC004944', 'AC004947', 'AC005019', 'AC005050', 'AC005144', 'AC005150', 'AC005165', 'AC005186', 'AC005304', 'AC005307', 'AC005487', 'AC005523', 'AC005616', 'AC005699', 'AC006004', 'AC006058', 'AC006065', 'AC006148', 'AC006305', 'AC006504', 'AC006978', 'AC007064', 'AC007126', 'AC007325', 'AC007344', 'AC007364', 'AC007370', 'AC007388', 'AC007389', 'AC007423', 'AC007495', 'AC007611', 'AC007639', 'AC007681', 'AC007953', 'AC008033', 'AC008035', 'AC008109', 'AC008268', 'AC008549', 'AC008632', 'AC008663', 'AC008691', 'AC008703', 'AC008780', 'AC008825', 'AC008875', 'AC008892', 'AC009075', 'AC009081', 'AC009093', 'AC009226', 'AC009262', '

/usr/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [65]:
cur_data.adata.var

,gene_name,ensembl_gene_id,gene_symbol,original_index
index,,,,
0,A1BG,ENSG00000121410,A1BG,0
1,A1BG-AS1,ENSG00000268895,A1BG-AS1,1
2,A1CF,ENSG00000148584,A1CF,2
3,A2M,ENSG00000175899,A2M,3
4,A2M-AS1,ENSG00000245105,A2M-AS1,4
...,...,...,...,...
23707,ZXDC,ENSG00000070476,ZXDC,23707
23708,ZYG11A,ENSG00000203995,ZYG11A,23708
23709,ZYG11B,ENSG00000162378,ZYG11B,23709


### Validate var metadata

In [66]:
cur_data.validate_data(slot='var')

,ensembl_gene_id,gene_symbol
index,,
0,ENSG00000121410,A1BG
1,ENSG00000268895,A1BG-AS1
2,ENSG00000148584,A1CF
3,ENSG00000175899,A2M
4,ENSG00000245105,A2M-AS1
...,...,...
23707,ENSG00000070476,ZXDC
23708,ENSG00000203995,ZYG11A
23709,ENSG00000162378,ZYG11B


# Save the dataset

In [67]:
cur_data.save_curated_data_h5ad()

/content/PerturbationCatalogue/data_exploration/curation_tools/curation_tools.py:327: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  adata.obs = adata.obs.fillna(value=np.nan)


✅ Curated h5ad data saved to ../curated/h5ad/frangieh_2021_raw_curated.h5ad


In [68]:
cur_data.save_curated_data_parquet(split_metadata=True, save_metadata_only=True)

✅ Metadata saved to ../curated/parquet/frangieh_2021_raw_curated_metadata.parquet


# Upload to BigQuery

In [69]:
upload_parquet_to_bq(
    parquet_path='../curated/parquet/frangieh_2021_raw_curated_metadata.parquet',
    bq_dataset_id='prj-ext-dev-pertcat-437314.perturb_seq',
    bq_table_name='metadata',
    key_columns=['dataset_id', 'sample_id'],
    verbose=True
)

Staging table: loading `.parquet` file ../curated/parquet/frangieh_2021_raw_curated_metadata.parquet to prj-ext-dev-pertcat-437314.perturb_seq.metadata_staging...
Staging table: loaded 126966 rows to prj-ext-dev-pertcat-437314.perturb_seq.metadata_staging
Staging table: added ingested_at timestamp column to prj-ext-dev-pertcat-437314.perturb_seq.metadata_staging
Merge completed: staging → prj-ext-dev-pertcat-437314.perturb_seq.metadata with type-safe casting.
Staging table: deleted prj-ext-dev-pertcat-437314.perturb_seq.metadata_staging


# Upload to GC Storage

In [70]:
!gcloud storage cp ../curated/h5ad/frangieh_2021_raw_curated.h5ad gs://perturbation-catalogue-lake/perturbseq/curated/

uploading large objects. If you would like to opt-out and instead
perform a normal upload, run:
`gcloud config set storage/parallel_composite_upload_enabled False`
If you would like to disable this warning, run:
`gcloud config set storage/parallel_composite_upload_enabled True`
Note that with parallel composite uploads, your object might be
uploaded as a composite object
(https://cloud.google.com/storage/docs/composite-objects), which means
that any user who downloads your object will need to use crc32c
checksums to verify data integrity. gcloud storage is capable of
computing crc32c checksums, but this might pose a problem for other
clients.

Copying file://../curated/h5ad/frangieh_2021_raw_curated.h5ad to gs://perturbation-catalogue-lake/perturbseq/curated/frangieh_2021_raw_curated.h5ad

Average throughput: 881.2MiB/s
